# Phase 1: Environment Setup & Data Ingestion
* **Objective:** Initialize the workspace, suppress execution warnings, and load the localized spatial-temporal traffic datasets.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")

Loading datasets...
Train shape : (77299, 11)
Test shape  : (41778, 10)


# Phase 2: Structural Preprocessing & Feature Engineering
* **Temporal High-Res Grid:** Extracts precise 15-minute resolution cycles using continuous Sine/Cosine transformations.
* **Categorical Constraints:** Directs non-numeric columns (`RoadType`, `Weather`, etc.) into raw string formats optimized for CatBoost's native split-finding algorithms.

In [2]:
print("Constructing Time-Series Sequence and Momentum Features...")

# 1. Combine sets to allow Day 48 traffic momentum to flow into Day 49
train_df['is_test'] = 0
test_df['is_test'] = 1
test_df['demand'] = np.nan # Test set has no demand yet

df = pd.concat([train_df, test_df], ignore_index=True)

# 2. Extract Temporal Grids
df['Hour'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]))
df['Minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]))
df['total_minutes'] = df['day'] * 24 * 60 + df['Hour'] * 60 + df['Minute']

# High-resolution cyclical encoding
df['Time_sin'] = np.sin(2 * np.pi * df['total_minutes'] / (24 * 60))
df['Time_cos'] = np.cos(2 * np.pi * df['total_minutes'] / (24 * 60))

# 3. CRITICAL: Sort chronologically within each geohash
df = df.sort_values(by=['geohash', 'total_minutes']).reset_index(drop=True)

# 4. Lag Features (What was traffic like 15m, 30m, and 60m ago in this exact spot?)
for lag in [1, 2, 4]: 
    df[f'demand_lag_{lag}'] = df.groupby('geohash')['demand'].shift(lag)

# 5. Rolling Features (Moving average of the last hour)
df['rolling_mean_4'] = df.groupby('geohash')['demand'].shift(1).rolling(window=4, min_periods=1).mean()

# 6. Clean Categoricals & Fill Missing Standard Features
CAT_COLS = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for col in CAT_COLS:
    df[col] = df[col].fillna('Unknown').astype(str)

df['Temperature'] = df['Temperature'].fillna(train_df['Temperature'].median())

# 7. Split back into proper Train and Test structures
train_engineered = df[df['is_test'] == 0].drop(columns=['is_test', 'timestamp', 'Index'])
test_engineered = df[df['is_test'] == 1].drop(columns=['is_test', 'timestamp', 'demand'])

print(f"Feature matrix generated with {len(train_engineered.columns)} columns.")

Constructing Time-Series Sequence and Momentum Features...
Feature matrix generated with 18 columns.


In [3]:
print("Applying chronological split and training CatBoost...")

X = train_engineered.drop(columns=['demand'])
y = train_engineered['demand']
X_test_final = test_engineered.copy()

# 1. TIME-SERIES SPLIT: shuffle=False is absolutely mandatory here. 
# We must train on the past to predict the future.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42, shuffle=False)

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
overall_mean = y_train_log.mean()

# 2. Learn the Historical Baseline (Only from Train)
train_enc = X_train.copy()
train_enc['target_log'] = y_train_log
geo_mean = train_enc.groupby('geohash')['target_log'].mean()
geo_prefix4_mean = train_enc.assign(prefix=train_enc['geohash'].str[:4]).groupby('prefix')['target_log'].mean()

# 3. Dynamic Imputation Function
def impute_lags_and_encode(df_in):
    df_out = df_in.copy()
    # Create the neighborhood fallback safety net
    df_out['Geo_Mean_Log'] = df_out['geohash'].map(geo_mean).fillna(
        df_out['geohash'].str[:4].map(geo_prefix4_mean)
    ).fillna(overall_mean)
    
    # If a lag feature is missing (e.g., the very first row of the day), fill it with the historical average
    for lag in [1, 2, 4]:
        df_out[f'demand_lag_{lag}'] = df_out[f'demand_lag_{lag}'].fillna(np.expm1(df_out['Geo_Mean_Log']))
    
    df_out['rolling_mean_4'] = df_out['rolling_mean_4'].fillna(np.expm1(df_out['Geo_Mean_Log']))
    
    # Drop the temporary log map as CatBoost handles categoricals natively
    return df_out.drop(columns=['Geo_Mean_Log'])

# Apply imputation safely
X_train = impute_lags_and_encode(X_train)
X_val = impute_lags_and_encode(X_val)
X_test_final = impute_lags_and_encode(X_test_final)

# 4. Build Data Pools & Train
train_pool = Pool(X_train, y_train_log, cat_features=CAT_COLS)
val_pool   = Pool(X_val, y_val_log, cat_features=CAT_COLS)
test_pool  = Pool(X_test_final, cat_features=CAT_COLS)

model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.04,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    eval_metric='RMSE',
    od_type='Iter',
    od_wait=200,
    verbose=250
)

model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# 5. Local Validation Check
val_preds = np.expm1(model.predict(val_pool))
y_val_act = np.expm1(y_val_log)

print("\n=== True Time-Series Validation Performance ===")
print(f"R2 Score : {r2_score(y_val_act, np.clip(val_preds, 0, None)) * 100:.2f}%")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_val_act, np.clip(val_preds, 0, None))):.5f}")

Applying chronological split and training CatBoost...
0:	learn: 0.1090060	test: 0.0790921	best: 0.0790921 (0)	total: 673ms	remaining: 33m 39s
250:	learn: 0.0222508	test: 0.0253988	best: 0.0238976 (111)	total: 2m 10s	remaining: 23m 53s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.0238975618
bestIteration = 111

Shrink model to first 112 iterations.

=== True Time-Series Validation Performance ===
R2 Score : 91.76%
RMSE     : 0.02691


In [4]:
print("Generating momentum-based final predictions...")

test_preds_log = model.predict(test_pool)
test_preds = np.expm1(test_preds_log)
test_preds = np.clip(test_preds, 0, None)

submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_preds
})

submission.to_csv('catboost_momentum_submission.csv', index=False)
print("Submission file 'catboost_momentum_submission.csv' generated successfully!")

Generating momentum-based final predictions...
Submission file 'catboost_momentum_submission.csv' generated successfully!
